# 01 · Ingesta y limpieza

**Etapa 01 del pipeline** (ingestión y primera transformación). Lee los dos datasets crudos de la Parte I,
valida su estructura, convierte los textos a tipos, recupera el identificador de cada oferta, deduplica y
separa las publicaciones que no son el componente buscado.

| | Archivo | Rol |
| --- | --- | --- |
| Entrada | `data/ec1_homeoffice/homeoffice_ml_crudo.csv` | dataset inicial principal (home office, 15/09/2026) — solo lectura |
| Entrada | `data/crudos/mercadolibre_raw.csv` | dataset inicial secundario (categorías, 13/09/2026) — solo lectura |
| Salida | `data/intermedio/01_homeoffice_limpio.csv` | home office limpio, una fila por oferta |
| Salida | `data/intermedio/01_categorias_limpio.csv` | categorías limpio, para la integración (notebook 03) |
| Evidencia | `data/evidencia/01_*.csv` | inconsistencias, duplicados, publicaciones ajenas y reasignadas |

**Técnicas del curso**: limpieza de datos nivel 1 (estructura estándar, nombres codificables, identificador
único por fila) y nivel 2 (reestructuración y tipado), de la sesión U3_L2. Las decisiones de esta etapa están
en `specs/001-preprocesamiento-analisis-home-office/research.md` (R1 y R4).

In [1]:
import re
from urllib.parse import urlsplit, parse_qs

import pandas as pd
import matplotlib.pyplot as plt

import comun
from comun import (CRUDO_HO, CRUDO_CAT, INTERMEDIO, EVIDENCIA, RAIZ, escribir_csv, guardar_evidencia,
                   registrar, registrar_balance)

comun.aplicar_estilo()
pd.set_option('display.max_colwidth', 90)

## 1. Ingestión y validación de estructura (nivel 1)

Se leen ambos archivos **como texto**, para no perder ceros ni formatos antes de decidir cada conversión, y se
verifica que traigan exactamente las columnas que produjo el scraper. Si el esquema cambia, el notebook se
detiene aquí (FR-002, FR-026). `fila_csv` guarda la línea del archivo original para rastrear cualquier registro.

In [2]:
COLUMNAS_HO = ['categoria', 'pagina', 'posicion_pagina', 'id_publicacion', 'titulo', 'moneda',
               'precio_actual_texto', 'precio_anterior_texto', 'descuento_texto', 'cuotas_texto',
               'vendedor_texto', 'envio_texto', 'resenas_texto', 'etiqueta_texto', 'url_publicacion',
               'url_fuente', 'fecha_extraccion']
COLUMNAS_CAT = ['categoria', 'pagina', 'posicion_pagina', 'id_publicacion', 'titulo', 'moneda',
                'precio_actual_texto', 'precio_anterior_texto', 'cuotas_texto', 'vendedor_texto',
                'url_publicacion', 'url_fuente', 'fecha_extraccion']

crudo_ho = pd.read_csv(CRUDO_HO, encoding='utf-8-sig', dtype='string')
crudo_cat = pd.read_csv(CRUDO_CAT, encoding='utf-8-sig', dtype='string')

for nombre, df, esperadas in [('home office', crudo_ho, COLUMNAS_HO), ('categorías', crudo_cat, COLUMNAS_CAT)]:
    assert list(df.columns) == esperadas, f'Esquema inesperado en {nombre}: {list(df.columns)}'
    assert all(re.fullmatch(r'[a-z][a-z0-9_]*', c) for c in df.columns), f'Nombres no codificables en {nombre}'
    print(f'{nombre:12s} {df.shape[0]:>5} filas × {df.shape[1]} columnas — esquema y nombres válidos')

ho = crudo_ho.copy()
ho.insert(0, 'fila_csv', range(2, len(ho) + 2))
for c in COLUMNAS_HO:
    ho[c] = ho[c].str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA)

N_CRUDO_HO = len(ho)
registrar('ingesta', filas_crudo_ho=N_CRUDO_HO, columnas_crudo_ho=crudo_ho.shape[1],
          filas_crudo_cat=len(crudo_cat), columnas_crudo_cat=crudo_cat.shape[1])
ho.head(3)

home office   3680 filas × 17 columnas — esquema y nombres válidos
categorías    5080 filas × 13 columnas — esquema y nombres válidos


,fila_csv,categoria,pagina,posicion_pagina,id_publicacion,titulo,moneda,precio_actual_texto,precio_anterior_texto,descuento_texto,cuotas_texto,vendedor_texto,envio_texto,resenas_texto,etiqueta_texto,url_publicacion,url_fuente,fecha_extraccion
0,2,laptops,1,1,MPE64140599,"Laptop HP 15-fd0255la, Intel Core i5, 8 GB RAM, 512 GB SSD, 15.6"" FHD, FreeDOS Dorado",S/,S/ 2.199,S/ 2.999,26% OFF,"18 cuotas de S/ 122 , 17 sin interés",Mercado Libre,Llega gratis mañana,4.7 | +50 vendidos,<NA>,https://www.mercadolibre.com.pe/laptop-hp-15-fd0255la-intel-core-i5-8-gb-ram-512-gb-ss...,https://listado.mercadolibre.com.pe/laptops,2026-09-15 23:34:39
1,3,laptops,1,2,MPE65867513,Lenovo Loq 15iax9 Core I5 12450hx 16gb 512gb Rtx3050 6gb Gris,S/,S/ 3.299,S/ 3.890,15% OFF,"6 cuotas de S/ 549 , 83 sin interés",PC YA Empresarial,Llega gratis mañana,4.8 | +25 vendidos,<NA>,https://www.mercadolibre.com.pe/lenovo-loq-15iax9-core-i5-12450hx-16gb-512gb-rtx3050-6...,https://listado.mercadolibre.com.pe/laptops,2026-09-15 23:34:39
2,4,laptops,1,3,MPE66346869,Asus Tuf Gaming F16 (2025) Core I7 14650hx 32gb 1tb Rtx5060 Jaeger Gray,S/,S/ 6.699,S/ 10.199,34% OFF,6 cuotas de S/ 1.116 sin interés,LaptopShop,Llega gratis mañana,4.7 | +25 vendidos,<NA>,https://www.mercadolibre.com.pe/asus-tuf-gaming-f16-2025-core-i7-14650hx-32gb-1tb-rtx5...,https://listado.mercadolibre.com.pe/laptops,2026-09-15 23:34:39


## 2. Tipado de los textos (nivel 2)

El scraper guardó los valores tal como se ven en la tarjeta. Cada formato se convierte con una expresión
regular **anclada**: un texto que no encaja no se adivina, se registra como inconsistencia (FR-003).

| Texto crudo | Ejemplo | Variables |
| --- | --- | --- |
| precio actual y anterior | `S/ 2.199`, `S/ 987,90` | `precio`, `precio_anterior` (punto de miles, coma decimal) |
| descuento | `26% OFF` | `descuento_anunciado_pct` |
| cuotas | `18 cuotas de S/ 122 , 17 sin interés` | `numero_cuotas`, `monto_cuota` |
| reseñas | `4.8 \| +100 vendidos`, `+50 vendidos`, `4.9 \| +5mil vendidos` | `calificacion`, `vendidos_min` |
| envío | `Llega gratis mañana` | `envio` |

`vendidos_min` es una **cota inferior**: "+100 vendidos" significa al menos 100. La Parte I interpretó este
texto como número de opiniones y dejó la columna `opiniones` 100 % vacía; aquí se corrige.

In [3]:
def precio_peruano(serie):
    """'S/ 1.234,5' -> 1234.5 (punto de miles, coma decimal); lo que no encaja queda nulo."""
    numero = serie.str.replace(' ', '', regex=False).str.extract(
        r'^S/((?:\d{1,3}(?:\.\d{3})+|\d+)(?:,\d{1,2})?)$', expand=False)
    return pd.to_numeric(numero.str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
                         errors='coerce')


ho['precio'] = precio_peruano(ho['precio_actual_texto'])
ho['precio_anterior'] = precio_peruano(ho['precio_anterior_texto'])
ho['descuento_anunciado_pct'] = pd.to_numeric(
    ho['descuento_texto'].str.extract(r'^(\d{1,2})% OFF$', expand=False), errors='coerce').astype('Int64')

cuotas = ho['cuotas_texto'].str.extract(r'^(\d+) cuotas? de S/ ((?:\d{1,3}(?:\.\d{3})+|\d+)(?: , \d{1,2})?)'
                                        r' sin interés$')
ho['numero_cuotas'] = pd.to_numeric(cuotas[0], errors='coerce').astype('Int64')
ho['monto_cuota'] = precio_peruano('S/ ' + cuotas[1].str.replace(' , ', ',', regex=False))

resenas = ho['resenas_texto'].str.extract(r'^(?:(\d\.\d) \| )?\+(\d+)(mil)? vendidos$')
ho['calificacion'] = pd.to_numeric(resenas[0], errors='coerce')
ho['vendidos_min'] = (pd.to_numeric(resenas[1], errors='coerce')
                      * resenas[2].map({'mil': 1000}).fillna(1)).astype('Int64')

ENVIO = {'Llega gratis mañana': 'gratis mañana', 'Envío gratis Envío gratis': 'gratis',
         'Llega mañana': 'mañana con costo'}
ho['envio'] = ho['envio_texto'].map(ENVIO).astype('string')
ho['vendedor'] = ho['vendedor_texto']
ho['pagina'] = pd.to_numeric(ho['pagina']).astype('Int64')
ho['posicion_pagina'] = pd.to_numeric(ho['posicion_pagina']).astype('Int64')
ho['fecha_extraccion'] = pd.to_datetime(ho['fecha_extraccion'], format='%Y-%m-%d %H:%M:%S')

CONVERSIONES = {'precio_actual_texto': 'precio', 'precio_anterior_texto': 'precio_anterior',
                'descuento_texto': 'descuento_anunciado_pct', 'cuotas_texto': 'numero_cuotas',
                'resenas_texto': 'vendidos_min', 'envio_texto': 'envio'}
cobertura = pd.DataFrame([{'texto': t, 'variable': v, 'textos': int(ho[t].notna().sum()),
                           'convertidos': int(ho[v].notna().sum()),
                           'no_interpretables': int((ho[t].notna() & ho[v].isna()).sum())}
                          for t, v in CONVERSIONES.items()])
no_interpretables = pd.concat([ho.loc[ho[t].notna() & ho[v].isna()].assign(motivo=f'texto no interpretable: {t}')
                               for t, v in CONVERSIONES.items()])
assert ho['moneda'].eq('S/').all(), 'Aparece una moneda distinta de soles.'
assert ho['fecha_extraccion'].notna().all()
cobertura

,texto,variable,textos,convertidos,no_interpretables
0,precio_actual_texto,precio,3680,3680,0
1,precio_anterior_texto,precio_anterior,2020,2020,0
2,descuento_texto,descuento_anunciado_pct,2020,2020,0
3,cuotas_texto,numero_cuotas,3537,3537,0
4,resenas_texto,vendidos_min,1456,1456,0
5,envio_texto,envio,3444,3444,0


Todos los textos presentes se convirtieron: la tabla muestra cero no interpretables. Las expresiones son
estrictas a propósito; si una nueva captura trae otro formato, aparecerá aquí en lugar de convertirse mal.
La moneda es siempre `S/` y todas las fechas tienen el formato esperado.

## 3. Identificador de la oferta (R1)

Mercado Libre distingue el **producto de catálogo** (`/p/MPE…`, compartido por varios vendedores) de la
**oferta** concreta (parámetro `wid` de la URL). La Parte I identificó el dataset de categorías por oferta,
pero el de home office por catálogo. Aquí ambos se identifican por oferta, con la misma regla:

1. el parámetro `wid` de la URL;
2. si falta, el código de una ruta directa de publicación (sin `/p/`);
3. si tampoco existe, el código de catálogo con el prefijo `CAT-`, para que nunca se confunda con una oferta.

Las URL de `click1.mercadolibre.com.pe/mclics` son **anuncios patrocinados**: se marcan en `es_publicidad`.

In [4]:
def oferta_id(url):
    if pd.isna(url):
        return pd.NA
    partes = urlsplit(url)
    wid = parse_qs(partes.query + '&' + partes.fragment).get('wid', [''])[0]
    if re.fullmatch(r'MPE\d+', wid):
        return wid
    directo = re.search(r'/MPE-?(\d+)(?:-|/|$)', partes.path)
    if directo and '/p/' not in partes.path and 'mclics' not in partes.netloc + partes.path:
        return 'MPE' + directo.group(1)
    return pd.NA


ho['id_catalogo'] = ho['id_publicacion']
ho['id_oferta'] = ho['url_publicacion'].map(oferta_id).astype('string')
sin_oferta = ho['id_oferta'].isna()
ho.loc[sin_oferta, 'id_oferta'] = 'CAT-' + ho.loc[sin_oferta, 'id_catalogo']
ho['es_publicidad'] = ho['url_publicacion'].str.contains('/mclics/', regex=False)

origen = pd.Series(['wid'] * len(ho), index=ho.index).where(~sin_oferta, 'catálogo (respaldo)')
print(origen.value_counts().to_string(), '\n')
print('Anuncios patrocinados:', int(ho['es_publicidad'].sum()), 'de', len(ho))

# Identificación de entidades: ¿cuántas ofertas aparecen bajo más de un código de catálogo?
codigos = ho.groupby('id_oferta')['id_catalogo'].nunique()
ofertas_multicodigo = codigos[codigos > 1]
print('Ofertas que aparecen con más de un código de catálogo:', len(ofertas_multicodigo))
ho[ho['id_oferta'].isin(ofertas_multicodigo.index[:2])][['id_oferta', 'id_catalogo', 'titulo', 'precio']]

wid                    3660
catálogo (respaldo)      20 

Anuncios patrocinados: 800 de 3680
Ofertas que aparecen con más de un código de catálogo: 43


,id_oferta,id_catalogo,titulo,precio
2358,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2406,MPE1068824028,MPE66433449,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2424,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2492,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2560,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2628,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2696,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2764,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2832,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0
2900,MPE1068824028,MPE1068824028,Mouse Inalámbrico De Hello Kitty Romax Rosa,39.0


Una misma oferta aparece con códigos de catálogo distintos cuando el sitio la muestra a veces como página de
producto y a veces como publicación directa. Deduplicar por catálogo, como hizo la Parte I, deja esas ofertas
repetidas.

## 4. Inconsistencias entre campos relacionados (FR-004)

Se comprueban tres relaciones que deben cumplirse en una tarjeta coherente. Las inconsistencias **no se
borran aquí**: se registran como evidencia y el notebook 02 decide su tratamiento junto con los atípicos.

- El precio anterior debe ser mayor que el precio actual.
- El descuento anunciado debe coincidir con `(precio_anterior − precio) / precio_anterior` (tolerancia de 1,5
  puntos, porque el sitio redondea).
- Las cuotas son "sin interés": `numero_cuotas × monto_cuota` debe ser igual al precio (tolerancia de 2 %).

In [5]:
desc_calculado = (ho['precio_anterior'] - ho['precio']) / ho['precio_anterior'] * 100
total_cuotas = ho['numero_cuotas'].astype('Float64') * ho['monto_cuota']
reglas = {
    'precio anterior no mayor que el precio': ho['precio_anterior'].notna() & (ho['precio_anterior'] <= ho['precio']),
    'descuento anunciado distinto del calculado': (ho['descuento_anunciado_pct'].notna()
                                                    & ((desc_calculado - ho['descuento_anunciado_pct']).abs() > 1.5)),
    'precio anterior sin descuento o descuento sin precio anterior': (ho['precio_anterior'].isna()
                                                                       != ho['descuento_anunciado_pct'].isna()),
    'cuotas que no suman el precio': total_cuotas.notna() & ((total_cuotas / ho['precio'] - 1).abs() > 0.02),
}
inconsistencias = pd.concat([ho.loc[mascara.fillna(False)].assign(motivo=motivo) for motivo, mascara in reglas.items()]
                            + [no_interpretables])
resumen_inc = inconsistencias['motivo'].value_counts().rename('filas').to_frame()
print(resumen_inc.to_string())
cols_inc = ['id_oferta', 'fila_csv', 'categoria', 'titulo', 'precio', 'precio_anterior', 'descuento_anunciado_pct',
            'numero_cuotas', 'monto_cuota']
guardar_evidencia(inconsistencias[cols_inc + ['motivo']].rename(columns={'motivo': 'detalle'}),
                  '01_inconsistencias.csv', '01', 'inconsistencia registrada (se conserva)')
inconsistencias.loc[inconsistencias['motivo'] != 'descuento anunciado distinto del calculado', cols_inc].head()

                                            filas
motivo                                           
descuento anunciado distinto del calculado     34
precio anterior no mayor que el precio          9


,id_oferta,fila_csv,categoria,titulo,precio,precio_anterior,descuento_anunciado_pct,numero_cuotas,monto_cuota
19,MPE730655082,21,laptops,Laptop Hp Gamer Victus 15.6 Fhd 144hz 16gb Ram Ssd 1tb Ssd I5-13420h W11 Rtx 4050 Negro,5665.0,5586.0,20,6,944.28
22,MPE1093122102,24,laptops,Portatil Hp 15-fd0253la Intel Core I5 1334u 8gb 512gb Win11 Gris,3000.0,2800.0,12,6,500.0
55,MPE636207759,57,laptops,Asus Tuf A16 2025 Ryzen 7 7445hs 16gb 1tb Ssd Rtx4050 144hz Mecha Gray,7636.0,7465.0,27,6,1272.0
108,MPE960716710,110,laptops,Asus TUF Gaming F16 FX607VJ 2025 Core 5 210h 16gb 512gb Rtx3050 6gb 144hz Mecha Gray,5332.0,5211.0,21,6,888.7
1170,MPE1095123156,1172,sillas-de-oficina,Silla De Oficina Inius Viggo Basculante con Cabecera Gris,409.0,409.0,58,6,68.17


## 5. Deduplicación por oferta (FR-005)

Una oferta se repite cuando aparece en varias páginas o búsquedas. Se conserva la observación **más reciente**
y, en empate, la última fila del archivo. Se compara con la deduplicación por catálogo de la Parte I.

In [6]:
ho = ho.sort_values(['fecha_extraccion', 'fila_csv'], kind='stable')
repetida = ho.duplicated('id_oferta', keep='last')
duplicados = ho.loc[repetida]
ho = ho.loc[~repetida].sort_values('fila_csv').reset_index(drop=True)
guardar_evidencia(duplicados[['id_oferta', 'fila_csv', 'categoria', 'titulo', 'precio', 'fecha_extraccion']],
                  '01_duplicados.csv', '01', 'oferta repetida: se conserva la observación más reciente')

unicos_catalogo = crudo_ho['id_publicacion'].nunique()
comparacion = pd.DataFrame({'clave': ['catálogo (Parte I)', 'oferta (Parte II)'],
                            'publicaciones_unicas': [unicos_catalogo, len(ho)]})
print('Filas retiradas por repetición:', len(duplicados))
comparacion

Filas retiradas por repetición: 787


,clave,publicaciones_unicas
0,catálogo (Parte I),2936
1,oferta (Parte II),2893


## 6. Publicaciones que no son el componente buscado (FR-032, R4)

Una búsqueda devuelve también productos relacionados. Para el costo de un componente solo sirve **una unidad de
ese componente**. Un prototipo mostró que buscar palabras sueltas da falsos positivos ("switch" en 81 teclados
mecánicos, "cable" en audífonos con cable) y que muchos títulos de laptops y monitores no nombran el producto
("Lenovo Loq 15iax9 Core I5…"). La regla aplicada, en este orden:

| Paso | Regla | Tratamiento |
| --- | --- | --- |
| Accesorio | el título **empieza** con un sustantivo de accesorio o repuesto (funda, soporte, pad, almohadillas, ruedas, pistón, respaldo, mochila, pegatinas…) o es una alfombrilla sin el componente | se excluye |
| Combo | en teclado, mouse o audífonos, el título es un kit, combo o pack, o une dos periféricos (`teclado + mouse`), o trae una alfombrilla junto al componente | se excluye del costo; se analiza aparte en el notebook 05 |
| Otro producto | el título empieza con otro producto (tableta, iPad, parlante, cámara…) | se excluye |
| Reasignada | en búsquedas de periféricos, el título trae un procesador o una tarjeta gráfica (es una laptop), o no nombra su componente y el primer componente que nombra es otro (por ejemplo, un mouse en la búsqueda de teclados) | se cambia el componente; no se excluye |

Nunca se reasigna desde laptop o monitor: en sus títulos "teclado español" o "incluye mouse" describen el
equipo, no otro producto. Las reglas de repuestos, iPad y procesador se añadieron tras revisar uno por uno los
atípicos de precio del notebook 02: eran laptops, tabletas y repuestos de silla en búsquedas ajenas, no
errores de precio.

In [7]:
SUSTANTIVO = {
    'laptop': r'laptop|notebook|macbook|chromebook|ultrabook',
    'monitor': r'\bmonitor',
    'silla': r'\bsill[ao]n?\b|\bsila\b',
    'teclado': r'\bteclado|\bkeyboard',
    'mouse': r'\bmouse\b|\brat[oó]n\b',
    'audifonos': r'aud[ií]fono|auricular|headset|earbud|airpods|handsfree|manos libres|in-ear|over-ear|\bcascos?\b',
}
ACCESORIO_INICIO = (r'^(funda|soporte|base|mochila|malet[ií]n|cargador|cable|adaptador|protector|estuche|'
                    r'almohadillas?|repuesto|ruedas?|coj[ií]n|reposapi[eé]s|apoyapi[eé]s|stickers?|keycaps?|'
                    r'mouse ?pad|pad|alfombrilla|limpiador|brazo|hub|docking|cooler|bolso|organizador|'
                    r'respaldar|respaldo|pist[oó]n|apoyabrazos|brazos|cilindro|pegatinas?|calcoman[ií]as?|'
                    r'forro|cubierta)\b')
ALFOMBRILLA = r'mouse ?pad|alfombrilla|\bmousepad\b'
COMBO = (r'\b(kit|combo|pack)\b.*(teclado|mouse|rat[oó]n|aud[ií]fono)'
         r'|(teclado|mouse|rat[oó]n)\s*(\+|y|&|con)\s*(mouse|rat[oó]n|teclado)'
         r'|aud[ií]fonos?.*\+.*aud[ií]fonos?')
OTRO_PRODUCTO = r'^(tableta|tablet|ipad|parlante|c[aá]mara|micr[oó]fono|webcam|escritorio|mesa|impresora|router)\b'
# Procesador o tarjeta gráfica en el título: es una laptop aunque no diga "laptop".
FIRMA_LAPTOP = r'\b(core\s?i[3579]|core\s?ultra|ryzen\s?\d|rtx\s?\d{4}|celeron|intel\s?n\d{3})\b'
PERIFERICOS = {'teclado', 'mouse', 'audifonos', 'silla'}

ho['busqueda'] = ho['categoria']
ho['componente'] = ho['categoria'].map(comun.COMPONENTES)
assert ho['componente'].notna().all(), 'Búsqueda sin componente asignado.'


def primer_componente(titulo):
    posiciones = {c: m.start() for c, p in SUSTANTIVO.items() if (m := re.search(p, titulo))}
    return min(posiciones, key=posiciones.get) if posiciones else None


def clasificar(fila):
    t = fila['titulo'].lower()
    nombra_propio = bool(re.search(SUSTANTIVO[fila['componente']], t))
    if re.search(ACCESORIO_INICIO, t) or (re.search(ALFOMBRILLA, t) and not nombra_propio):
        return 'accesorio', None
    if fila['componente'] in ('teclado', 'mouse', 'audifonos') and (re.search(COMBO, t) or re.search(ALFOMBRILLA, t)):
        return 'combo de varios componentes', None
    if re.search(OTRO_PRODUCTO, t):
        return 'otro producto', None
    if fila['componente'] in PERIFERICOS and re.search(FIRMA_LAPTOP, t):
        return 'reasignada', 'laptop'
    if fila['componente'] in PERIFERICOS and not nombra_propio:
        primero = primer_componente(t)
        if primero and primero != fila['componente']:
            return 'reasignada', primero
    return None, None


clase = ho.apply(clasificar, axis=1, result_type='expand').rename(columns={0: 'motivo', 1: 'nuevo_componente'})
ho = ho.join(clase)
print(pd.crosstab(ho['componente'], ho['motivo'].fillna('se conserva')).to_string())

motivo      accesorio  combo de varios componentes  otro producto  reasignada  se conserva
componente                                                                                
audifonos          11                            1              5          11          471
laptop              0                            0              0           0          466
monitor             0                            0              0           0          482
mouse               6                            2              0           0          475
silla               5                            0              0           0          470
teclado            11                          136              1           0          340


Muestra de cada caso, para revisar que la regla separa lo que debe separar:

In [8]:
lineas = []  # se imprime de una vez: así la salida del notebook no cambia entre ejecuciones
for motivo, grupo in ho[ho['motivo'].notna()].groupby('motivo'):
    lineas.append(f'\n— {motivo} ({len(grupo)})')
    for _, f in grupo.drop_duplicates('titulo').groupby('componente').head(3).iterrows():
        destino = f' → {f["nuevo_componente"]}' if pd.notna(f['nuevo_componente']) else ''
        lineas.append(f'   [{f["componente"]}{destino}] {comun.soles(f["precio"], 2):>13}  {f["titulo"][:85]}')
print('\n'.join(lineas))


— accesorio (33)
   [silla]      S/ 14,90  Respaldar Ergonómico Para Silla Oficina Auto Soporte Espalda Negro Poliéster
   [silla]      S/ 69,90  Apoyabrazos - Brazos Relax Para Silla - De Uso Universal.
   [silla]      S/ 50,00  Pistón Amortiguador A Gas Gamer Para Silla Oficina Giratoria Negro
   [teclado]      S/ 39,00  Mouse Pad gamer Alfombrilla Ratón diseño Excel Grande XXL 80x30 Yijun
   [teclado]      S/ 58,34  Keycaps Redragon Scarab A130 Sp Pudding Negro Esp Cherry F
   [teclado]      S/ 59,00  Cable Para Teclado Redragon Coiled A115b Black Color Del Teclado Negro Idioma Español
   [mouse]      S/ 97,16  Base de carga para ratón inalámbrico para juegos Attack Shar, color negro
   [mouse]     S/ 128,00  Estuche Para Mouse Logitech Mx Master 4 Rígido Viajero
   [mouse]      S/ 55,95  Soporte para cable de mouse VSG Hyperion base antideslizante blanco
   [audifonos]     S/ 196,11  Mochila Para Viaje Impermeable Expandible 29l - 39l | Ebrand Color Negro Diseño De La
   [audifono

La revisión confirma la regla: los excluidos son ruedas, almohadillas, soportes, mochilas, kits de teclado y
mouse, tabletas digitalizadoras; los reasignados son mouse y laptops que aparecieron en otra búsqueda. Los
**combos** son frecuentes en teclados: el sitio los ofrece como alternativa al teclado solo, por eso se
guardan para compararlos en el análisis.

In [9]:
reasignadas = ho['motivo'].eq('reasignada')
guardar_evidencia(ho.loc[reasignadas, ['id_oferta', 'busqueda', 'componente', 'nuevo_componente', 'titulo', 'precio']],
                  '01_reasignadas.csv', '01', 'componente reasignado por el título (se conserva)')
ho.loc[reasignadas, 'componente'] = ho.loc[reasignadas, 'nuevo_componente']

excluir = ho['motivo'].notna() & ~reasignadas
ajenas = ho.loc[excluir]
guardar_evidencia(ajenas[['id_oferta', 'busqueda', 'componente', 'titulo', 'precio', 'numero_cuotas', 'monto_cuota']]
                  .rename(columns={'componente': 'componente_busqueda'}).assign(tipo=ajenas['motivo']),
                  '01_ajenas_al_componente.csv', '01', 'no es una unidad del componente buscado')
ho = ho.loc[~excluir].drop(columns=['motivo', 'nuevo_componente']).reset_index(drop=True)
print('Excluidas:', len(ajenas), '| reasignadas:', int(reasignadas.sum()), '| quedan:', len(ho))

Excluidas: 178 | reasignadas: 11 | quedan: 2715


## 7. Dataset de categorías: segunda fuente

El dataset de categorías ya se limpió en la Parte I. Para la integración se reconstruye desde su crudo con
**la misma regla** de esa etapa —filas sin campos esenciales, deduplicación por oferta y las 7 exclusiones de
precio documentadas en `data/calidad/precios_excluidos.csv`— y se comprueba que el resultado coincide oferta
por oferta con `data/limpios/mercadolibre_limpio.csv`. Así la integración parte de datos verificados y no de
una copia.

In [10]:
cat = crudo_cat.copy()
cat.insert(0, 'fila_csv', range(2, len(cat) + 2))
for c in COLUMNAS_CAT:
    cat[c] = cat[c].str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA)
cat['id_oferta'] = cat['url_publicacion'].map(oferta_id).astype('string')
cat['precio'] = precio_peruano(cat['precio_actual_texto'])
cat['pagina'] = pd.to_numeric(cat['pagina'], errors='coerce')
cat['posicion_pagina'] = pd.to_numeric(cat['posicion_pagina'], errors='coerce')
cat['fecha_extraccion'] = pd.to_datetime(cat['fecha_extraccion'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

invalida = (cat['id_oferta'].isna() | cat['titulo'].isna() | cat['categoria'].isna() | cat['precio'].isna()
            | cat['precio'].le(0) | cat['moneda'].ne('S/') | cat['fecha_extraccion'].isna()
            | cat['pagina'].isna() | cat['posicion_pagina'].isna())
cat_validas = cat.loc[~invalida].sort_values(['fecha_extraccion', 'fila_csv'], kind='stable')
cat_repetida = cat_validas.duplicated('id_oferta', keep='last')
cat_unicas = cat_validas.loc[~cat_repetida]
excluidos_parte1 = pd.read_csv(RAIZ / 'data' / 'calidad' / 'precios_excluidos.csv', dtype='string')['id_publicacion']
cat_final = cat_unicas.loc[~cat_unicas['id_oferta'].isin(excluidos_parte1)].sort_values('fila_csv')

limpio_parte1 = pd.read_csv(RAIZ / 'data' / 'limpios' / 'mercadolibre_limpio.csv', dtype='string')
assert set(cat_final['id_oferta']) == set(limpio_parte1['id_publicacion']), \
    'La reconstrucción no coincide con el dataset limpio de la Parte I.'
registrar_balance('01_categorias', len(cat), len(cat_final), invalidas=int(invalida.sum()),
                  duplicados=int(cat_repetida.sum()), precios_excluidos_parte1=int(cat_unicas['id_oferta']
                                                                                    .isin(excluidos_parte1).sum()))
print('Categorías: coincide oferta por oferta con la Parte I —', len(cat_final), 'ofertas')

cat_salida = (cat_final.rename(columns={'categoria': 'busqueda'})
              [['id_oferta', 'busqueda', 'titulo', 'precio', 'url_publicacion', 'fecha_extraccion']])
escribir_csv(cat_salida, INTERMEDIO / '01_categorias_limpio.csv');

Categorías: coincide oferta por oferta con la Parte I — 4412 ofertas


## 8. Exportación y verificación

Se conservan los textos que el notebook 02 necesita para diagnosticar los faltantes (`etiqueta_texto` incluida:
su eliminación es una decisión de esa etapa). Los demás textos crudos siguen disponibles en el CSV original.

In [11]:
COLUMNAS_SALIDA = ['id_oferta', 'id_catalogo', 'componente', 'busqueda', 'titulo', 'precio', 'precio_anterior',
                   'descuento_anunciado_pct', 'numero_cuotas', 'monto_cuota', 'calificacion', 'vendidos_min',
                   'envio', 'vendedor', 'etiqueta_texto', 'es_publicidad', 'pagina', 'posicion_pagina',
                   'url_publicacion', 'fecha_extraccion']
salida = ho[COLUMNAS_SALIDA]

assert salida['id_oferta'].is_unique and salida['id_oferta'].notna().all(), '`id_oferta` único y sin nulos'
assert salida['precio'].gt(0).all(), '`precio > 0`'
assert salida['componente'].isin(comun.ORDEN_COMPONENTES).all()
registrar_balance('01_homeoffice', N_CRUDO_HO, len(salida), duplicados=len(duplicados), ajenas=len(ajenas))
escribir_csv(salida, INTERMEDIO / '01_homeoffice_limpio.csv')

registrar('limpieza',
          ofertas_unicas=len(salida), unicos_por_catalogo_parte1=int(unicos_catalogo),
          duplicados=len(duplicados), ofertas_con_varios_codigos=len(ofertas_multicodigo),
          anuncios_patrocinados_crudo=int(crudo_ho['url_publicacion'].str.contains('/mclics/').sum()),
          oferta_por_respaldo_catalogo=int(sin_oferta.sum()),
          ajenas=ajenas['motivo'].value_counts().to_dict(), reasignadas=int(reasignadas.sum()),
          inconsistencias=resumen_inc['filas'].to_dict(), textos_no_interpretables=len(no_interpretables),
          ofertas_por_componente=salida['componente'].value_counts().reindex(comun.ORDEN_COMPONENTES).to_dict())
print('01_homeoffice_limpio.csv:', salida.shape)
salida['componente'].value_counts().reindex(comun.ORDEN_COMPONENTES).to_frame('ofertas')

01_homeoffice_limpio.csv: (2715, 20)


,ofertas
componente,
laptop,476
monitor,483
silla,470
teclado,340
mouse,475
audifonos,471


## Resultado de la etapa

- Ambos crudos cumplen el esquema esperado y todos sus textos se convirtieron sin casos no interpretables.
- El home office queda identificado por **oferta**, la misma clave que el dataset de categorías; la
  comparación con la clave de catálogo de la Parte I está arriba.
- Se registraron las inconsistencias entre campos para tratarlas en el notebook 02.
- Las publicaciones que no son una unidad del componente salieron del cálculo de costos y quedaron como
  evidencia; los combos se reutilizan en el análisis.
- El balance de registros cuadra y queda en `data/final/metricas.json`.